<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_02_Data_Analysis_with_Pandas_and_NumPy/Week_4_Data_Manipulation/Day_24_Apply_Map_and_Transform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 24: Apply, Map, and Transform

## Phase 1 | Month 2 | Week 4 | Day 24

**Goal:** Master function application in Pandas — from fast vectorised ops to complex row-level custom logic.

### What You Will Learn
1. `Series.map()` — element-wise with dict or function
2. `DataFrame.apply()` — row or column level
3. `GroupBy.transform()` — group-aware operations
4. When to use `apply` vs vectorised operations
5. Performance benchmarks

### Resources
- 📖 [Pandas Apply Reference](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)
- 🎥 [Apply vs Vectorisation — Towards Data Science](https://www.youtube.com/watch?v=3P6NMWWP4Q8)

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Series.map() — element-wise transformation
df = pd.DataFrame({
    'county': ['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'],
    'code'  : [1, 2, 3, 4, 5],
    'score' : [85, 72, 63, 91, 78],
})

# Using a dict
region_map = {'Nairobi':'Central','Mombasa':'Coast','Kisumu':'Nyanza','Nakuru':'Rift Valley','Eldoret':'Rift Valley'}
df['region'] = df['county'].map(region_map)
print(df)

# Using a lambda
df['score_pct'] = df['score'].map(lambda x: f'{x}%')
print(df[['county','score','score_pct']])

In [ ]:
import pandas as pd
import numpy as np
import time
np.random.seed(0)

df = pd.DataFrame({'a': np.random.randn(10_000), 'b': np.random.randn(10_000)})

# Row-level apply (axis=1) — SLOW but flexible
t = time.perf_counter()
df['hyp_apply'] = df.apply(lambda r: (r['a']**2 + r['b']**2)**0.5, axis=1)
apply_time = time.perf_counter() - t

# Vectorised alternative — FAST
t = time.perf_counter()
df['hyp_vec'] = np.sqrt(df['a']**2 + df['b']**2)
vec_time = time.perf_counter() - t

print(f'apply() time  : {apply_time*1000:.1f} ms')
print(f'vectorised    : {vec_time*1000:.2f} ms')
print(f'Speedup       : {apply_time/vec_time:.0f}x')
print(f'Results match : {df["hyp_apply"].round(6).equals(df["hyp_vec"].round(6))}')

print('\nRule: Use apply() ONLY when no vectorised equivalent exists!')

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# GroupBy transform — adds group stats back to original rows
n = 200
df = pd.DataFrame({
    'employee': range(1, n+1),
    'dept'    : np.random.choice(['IT','HR','Finance','Sales'], n),
    'salary'  : np.random.randint(60_000, 350_000, n),
})

# Add dept-level stats to each row
df['dept_mean_salary'] = df.groupby('dept')['salary'].transform('mean')
df['salary_vs_dept']   = df['salary'] - df['dept_mean_salary']
df['salary_pct_rank']  = df.groupby('dept')['salary'].transform(
    lambda x: x.rank(pct=True).round(2)
)

print('Employee dataset with group context:')
print(df.head(8).to_string())
print('\nHighest paid vs dept avg:')
print(df.nlargest(3, 'salary_vs_dept')[['dept','salary','dept_mean_salary','salary_vs_dept']])

## Exercise: Feature Engineering Pipeline

Build a feature engineering function that creates 10 new features.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

n = 500
df = pd.DataFrame({
    'cust_id'   : range(1, n+1),
    'age'       : np.random.randint(18, 75, n),
    'income'    : np.random.exponential(80_000, n),
    'dependents': np.random.randint(0, 6, n),
    'loan_amt'  : np.random.exponential(200_000, n),
    'loan_term_months': np.random.choice([12,24,36,48,60], n),
    'credit_score': np.random.randint(300, 850, n),
    'region'    : np.random.choice(['Nairobi','Coast','Rift Valley','Nyanza','Western'], n),
})

# Create features:
# 1. debt_to_income = loan_amt / income
# 2. monthly_payment = loan_amt / loan_term_months
# 3. age_group: Young (<30), Mid (30-50), Senior (>50)
# 4. income_per_dependent = income / (dependents + 1)
# 5. credit_tier: Poor(<500), Fair(500-650), Good(650-750), Excellent(>750)
# 6. region_avg_income (group transform)
# 7. income_vs_region (income - region_avg_income)
# 8. risk_score: rule-based using map/apply
# YOUR CODE HERE

## Summary

- `map()`: element-wise with dict or function (Series only)
- `apply(axis=0)`: column-wise function (default)
- `apply(axis=1)`: row-wise function (slow — use vectorised when possible)
- `transform()`: group-aware, returns same-shape DataFrame
- Golden rule: if you can vectorise it, do it; `apply` is a last resort

**Next: Day 25 — Reading and Writing Data Files**